# SVIES — Custom Model Training (GPU)

Trains two custom YOLOv8 detectors on Tesla V100 GPU:
1. **Plate Detector** — License plate detection
2. **Helmet Detector** — Helmet/no-helmet detection

In [1]:
import sys
!{sys.executable} -m pip install -q ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 138.5 MB/s eta 0:00:0000:01


In [2]:
import os
import torch
import sys

print(f"Python: {sys.version}")

# Use GPU 4 (nearly empty)
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA Available: False


In [ ]:
# ═══ CONFIGURATION ═══
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY_HERE")
WORKSPACE = "dip-zrgjd"

PLATE_PROJECT = "vehicle-registration-plates-trudk-14wpm"
PLATE_VERSION = 1

HELMET_PROJECT = "hard-hat-workers-68ds8"
HELMET_VERSION = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PLATE_EPOCHS = 50
HELMET_EPOCHS = 50
BATCH_SIZE = 32

print(f"Device: {DEVICE}")
print(f"Batch:  {BATCH_SIZE}")

## Download Datasets

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Download plate dataset
plate_ds = rf.workspace(WORKSPACE).project(PLATE_PROJECT).version(PLATE_VERSION).download("yolov8")
print(f"Plate dataset: {plate_ds.location}")

# Download helmet dataset
helmet_ds = rf.workspace(WORKSPACE).project(HELMET_PROJECT).version(HELMET_VERSION).download("yolov8")
print(f"Helmet dataset: {helmet_ds.location}")

## Train Plate Detector

In [5]:
from ultralytics import YOLO
from pathlib import Path
import shutil

plate_dir = Path(plate_ds.location)
plate_yaml = plate_dir / "data.yaml"
if not plate_yaml.exists():
    plate_yaml = list(plate_dir.rglob("data.yaml"))[0]

model = YOLO("yolov8n.pt")
plate_results = model.train(
    data=str(plate_yaml),
    epochs=PLATE_EPOCHS,
    imgsz=640,
    device=DEVICE,
    batch=BATCH_SIZE,
    project="training_output",
    name="plate_detector",
    exist_ok=True,
    patience=15,
    workers=4,
)
print("Plate training complete!")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vehicle-Registration-Plates-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

KeyboardInterrupt: 

In [ ]:
# Save plate model
plate_best = Path(plate_results.save_dir) / "weights" / "best.pt"
plate_last = Path(plate_results.save_dir) / "weights" / "last.pt"
plate_src = plate_best if plate_best.exists() else plate_last
plate_out = Path("svies_plate_detector.pt")

shutil.copy2(str(plate_src), str(plate_out))
print(f"Saved: {plate_out} ({plate_out.stat().st_size / 1048576:.1f} MB)")

## Train Helmet Detector

In [ ]:
helmet_dir = Path(helmet_ds.location)
helmet_yaml = helmet_dir / "data.yaml"
if not helmet_yaml.exists():
    helmet_yaml = list(helmet_dir.rglob("data.yaml"))[0]

model2 = YOLO("yolov8n.pt")
helmet_results = model2.train(
    data=str(helmet_yaml),
    epochs=HELMET_EPOCHS,
    imgsz=416,
    device=DEVICE,
    batch=BATCH_SIZE,
    project="training_output",
    name="helmet_detector",
    exist_ok=True,
    patience=15,
    workers=4,
)
print("Helmet training complete!")

In [ ]:
# Save helmet model
helmet_best = Path(helmet_results.save_dir) / "weights" / "best.pt"
helmet_last = Path(helmet_results.save_dir) / "weights" / "last.pt"
helmet_src = helmet_best if helmet_best.exists() else helmet_last
helmet_out = Path("svies_helmet_detector.pt")

shutil.copy2(str(helmet_src), str(helmet_out))
print(f"Saved: {helmet_out} ({helmet_out.stat().st_size / 1048576:.1f} MB)")

## Validate & Summary

In [ ]:
print("=" * 60)
print("PLATE DETECTOR VALIDATION")
print("=" * 60)
pm = YOLO(str(plate_out))
pv = pm.val(data=str(plate_yaml), device=DEVICE)
print(f"  mAP50:    {pv.box.map50:.4f}")
print(f"  mAP50-95: {pv.box.map:.4f}")

print()
print("=" * 60)
print("HELMET DETECTOR VALIDATION")
print("=" * 60)
hm = YOLO(str(helmet_out))
hv = hm.val(data=str(helmet_yaml), device=DEVICE)
print(f"  mAP50:    {hv.box.map50:.4f}")
print(f"  mAP50-95: {hv.box.map:.4f}")

In [ ]:
print("=" * 60)
print("DONE")
print("=" * 60)
print(f"  svies_plate_detector.pt  ({plate_out.stat().st_size/1048576:.1f} MB)")
print(f"  svies_helmet_detector.pt ({helmet_out.stat().st_size/1048576:.1f} MB)")
print()
print("Copy both .pt files to your local svies/models/ folder.")
print("detector.py and helmet_detector.py will auto-load them.")